# Dispatch CSV — Source Data Analysis

**File:** `Output_of_.csv`  
**Purpose:** Document every assumption made before this file touches the dashboard.  
**Sections:**
1. Load raw file — shape, columns, dtypes, file size
2. Random sample — sanity check values
3. Column profiles — null rates, unique counts, top values
4. Required field validation
5. Cleaning steps — one at a time, row counts after each
6. Metric verification — reproduce every dashboard number by hand
7. Caveats — limitations and exclusions that affect interpretation

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── CONFIG: update this path if the file moves ────────────────────────
FILE_PATH = Path(r"D:\MullenAnalytics\ClientData\agencies\81d6f2e6-300a-48d0-b7b6-9250937d17fb\raw\7176dae1_Output_of_.csv")
assert FILE_PATH.exists(), f"File not found: {FILE_PATH}"

print(f"File size: {FILE_PATH.stat().st_size / 1_000_000:.2f} MB")

## 1. Load Raw File

In [ ]:
raw = pd.read_csv(FILE_PATH, low_memory=False)
print(f"Shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print()
print("Column names (as they appear in the file):")
for i, c in enumerate(raw.columns):
    print(f"  [{i:02d}] {c!r}  —  dtype: {raw[c].dtype}")

## 2. Random Sample — Do the values look right?

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 30)
pd.set_option('display.width', 200)
raw.sample(10, random_state=42)

## 3. Column Profiles — null rate, unique count, top 5 values

In [ ]:
def profile_col(s):
    null_pct = s.isnull().mean() * 100
    uniq     = s.nunique()
    top5     = s.value_counts(dropna=True).head(5).to_dict()
    return pd.Series({"null_%": f"{null_pct:.1f}", "unique": uniq, "top_5": top5})

profile = raw.apply(profile_col).T
profile

## 4. Required Field Validation

The pipeline requires four fields to process a dispatch file:
- **incident_number** (`Dispatch ID`)
- **date_created** (`Date Dispatched` — primary event timestamp)
- **incident_type** (`Patient Category` — note: `Call Type` is largely empty)
- **unit_id** (`Unit`)

Additionally, response-time metrics need:
- `Date Received` (call received → dispatch = call processing time)
- `Date Enroute` (dispatch → en route = turnout time)
- `Date Arrived` (en route → on scene = travel time)
- `Date Available` (on scene → clear = scene time)

In [ ]:
FIELD_MAP = {
    "incident_number":  "Dispatch ID",
    "date_dispatched":  "Date Dispatched",
    "date_received":    "Date Received",
    "date_enroute":     "Date Enroute",
    "date_arrived":     "Date Arrived",
    "date_available":   "Date Available",
    "incident_type":    "Patient Category",
    "unit_id":          "Unit",
    "municipality":     "Scene Grid",
    "priority":         "Dispatch Priority (Codetable)",
}

print(f"{'Semantic field':<22} {'Raw column':<35} {'Null %':>7} {'Parseable?':>10}")
print("-" * 80)
for sem, col in FIELD_MAP.items():
    if col not in raw.columns:
        print(f"{sem:<22} {'MISSING':35} {'—':>7} {'—':>10}")
        continue
    null_pct = raw[col].isnull().mean() * 100
    # Try datetime parse on timestamp fields
    if 'date' in sem.lower():
        parsed = pd.to_datetime(raw[col], errors='coerce')
        fail_pct = parsed.isnull().mean() * 100 - null_pct  # parse failures beyond natural nulls
        parseable = f"{max(0, 100 - fail_pct):.1f}%"
    else:
        parseable = "N/A"
    print(f"{sem:<22} {col:<35} {null_pct:>6.1f}% {parseable:>10}")

## 5. Cleaning Steps — one at a time

**Rule:** Print row count before and after each filter. Never silently drop rows.

In [ ]:
df = raw.copy()
print(f"START: {len(df):,} rows")

In [ ]:
# Step 1: Parse all timestamp columns
TS_COLS = ["Date Dispatched", "Date Received", "Date Enroute",
           "Date Arrived", "Date Available"]
for c in TS_COLS:
    df[c] = pd.to_datetime(df[c], errors='coerce')
print(f"After timestamp parse: {len(df):,} rows (no rows dropped — nulls set to NaT)")
print("\nNull counts after parse:")
for c in TS_COLS:
    print(f"  {c:<25}: {df[c].isnull().sum():,} null  ({df[c].isnull().mean()*100:.1f}%)")

In [ ]:
# Step 2: Drop rows with no dispatch timestamp (can't compute any metric without it)
before = len(df)
df = df.dropna(subset=["Date Dispatched"])
dropped = before - len(df)
print(f"Drop rows with null Date Dispatched: {before:,} → {len(df):,}  (dropped {dropped:,} rows, {dropped/before*100:.2f}%)")

In [ ]:
# Step 3: Compute phase interval seconds
# NOTE: Timestamps are MINUTE-resolution — all seconds digits are :00
# This means intervals can only be measured to ±60s precision.
df["call_processing_s"] = (df["Date Dispatched"] - df["Date Received"]).dt.total_seconds()
df["turnout_s"]         = (df["Date Enroute"]    - df["Date Dispatched"]).dt.total_seconds()
df["travel_s"]          = (df["Date Arrived"]    - df["Date Enroute"]).dt.total_seconds()
df["total_response_s"]  = (df["Date Arrived"]    - df["Date Received"]).dt.total_seconds()

# Verify minute-resolution limitation
sample_ts = df["Date Dispatched"].dropna().head(5)
print("Sample dispatch timestamps (confirming minute resolution):")
for t in sample_ts:
    print(f"  {t}  seconds digit = {t.second}")

In [ ]:
# Step 4: Bound each interval — drop physically impossible values
# Call processing: 0–3600s (calls processed in <0s or >60min are bad data)
# Turnout:         0–600s  (>10 min turnout is possible but check)
# Travel:          0–3600s
# Total response:  0–3600s (hard ceiling for BLS NFPA analysis)

for col, lo, hi, label in [
    ("call_processing_s", 0, 3600, "Call processing"),
    ("turnout_s",         0,  600, "Turnout"),
    ("travel_s",          0, 3600, "Travel"),
    ("total_response_s",  0, 3600, "Total response"),
]:
    valid = df[col].between(lo, hi)
    invalid = df[col].notna() & ~valid
    print(f"{label:<20}: {valid.sum():,} valid / {invalid.sum():,} out-of-bounds / {df[col].isna().sum():,} null (no arrival)")

In [ ]:
# Step 5: Flag cancelled calls (no Date Arrived)
# These are valid calls for VOLUME metrics but excluded from RESPONSE TIME metrics.
df["has_arrival"] = df["Date Arrived"].notna()
print(f"Calls with arrival time: {df['has_arrival'].sum():,} ({df['has_arrival'].mean()*100:.1f}%)")
print(f"Calls WITHOUT arrival (cancelled/mutual-aid/refused): {(~df['has_arrival']).sum():,} ({(~df['has_arrival']).mean()*100:.1f}%)")

## 6. Metric Verification

Reproduce every number that appears on the dashboard. For each metric, state:
- The formula
- The denominator
- Which rows are included

In [ ]:
# ── METRIC: Total call volume ────────────────────────────────────────────────
# Formula: count of rows after timestamp parse and dispatch-null drop
# Includes: all calls with a valid dispatch timestamp
# Excludes: rows where Date Dispatched could not be parsed
total_calls = len(df)
print(f"Total call volume: {total_calls:,}")
print(f"(pre-filter raw rows: {len(raw):,})")

In [ ]:
# ── METRIC: P90 total response time (all calls with valid arrival) ───────────
# Formula: 90th percentile of total_response_s where 0 <= value <= 3600
# Denominator: calls where Date Arrived is not null AND total_response_s in [0, 3600]
rt_valid = df["total_response_s"].between(0, 3600)
p90 = df.loc[rt_valid, "total_response_s"].quantile(0.90)
median_rt = df.loc[rt_valid, "total_response_s"].median()
mean_rt = df.loc[rt_valid, "total_response_s"].mean()
n_valid_rt = rt_valid.sum()

def fmt_sec(s):
    if pd.isna(s): return "—"
    m, r = divmod(round(s), 60)
    return f"{m}m {r:02d}s"

print(f"Response time denominator (valid arrivals, 0-3600s): {n_valid_rt:,} calls")
print(f"P90 total response:    {p90:.0f}s  ({fmt_sec(p90)})")
print(f"Median total response: {median_rt:.0f}s  ({fmt_sec(median_rt)})")
print(f"Mean total response:   {mean_rt:.0f}s  ({fmt_sec(mean_rt)})")

In [ ]:
# ── METRIC: NFPA 1710 compliance rate ───────────────────────────────────────
# Formula: % of calls where total_response_s <= 300 (5 minutes)
# Denominator: calls with valid total_response_s (same as above)
nfpa_target = 300
nfpa_pass = (df.loc[rt_valid, "total_response_s"] <= nfpa_target).mean() * 100
print(f"NFPA 1710 compliance: {nfpa_pass:.1f}% of {n_valid_rt:,} calls arrived in ≤{fmt_sec(nfpa_target)}")
print(f"NFPA 1710 target: ≥ 90%")
print(f"Status: {'PASS' if nfpa_pass >= 90 else 'FAIL'}")

In [ ]:
# ── METRIC: Phase interval medians ──────────────────────────────────────────
# Each phase is computed on its OWN valid subset — they do not sum to total response.
# This is intentional and should be noted in every UI that shows all four.
phases = [
    ("Call Processing (receive → dispatch)", "call_processing_s", 64),
    ("Turnout (dispatch → en route)",        "turnout_s",         60),
    ("Travel (en route → on scene)",         "travel_s",          240),
    ("Total Response (receive → on scene)",  "total_response_s",  300),
]

print(f"{'Phase':<45} {'N valid':>8}  {'Median':>8}  {'NFPA target':>12}  {'Status':>8}")
print("-" * 90)
for label, col, target in phases:
    valid_mask = df[col].between(0, 3600)
    n = valid_mask.sum()
    med = df.loc[valid_mask, col].median()
    status = "PASS" if med <= target else "FAIL"
    print(f"{label:<45} {n:>8,}  {fmt_sec(med):>8}  {fmt_sec(target):>12}  {status:>8}")

print()
print("NOTE: Phase medians computed on independent subsets. They cannot be summed.")

In [ ]:
# ── METRIC: Call volume by day of week ──────────────────────────────────────
# Formula: count of calls grouped by Day of Week of Dispatch
# The dashboard uses the pre-computed column, NOT derived from the timestamp.
# Verify they agree:
from_raw_col  = df["Day of Week of Dispatch"].value_counts().sort_index()
from_derived  = df["Date Dispatched"].dt.day_name().str[:3].str.upper().value_counts().sort_index()
print("From 'Day of Week of Dispatch' column:")
print(from_raw_col.to_string())
print()
# Note: day name abbreviation format may differ

In [ ]:
# ── METRIC: Call volume by incident type ────────────────────────────────────
# Using Patient Category (Call Type column is largely null — confirm below)
print("Call Type null rate:      ", raw["Call Type"].isnull().mean() * 100, "%")
print("Patient Category null rate:", raw["Patient Category"].isnull().mean() * 100, "%")
print()
print("Top 10 Patient Categories:")
print(df["Patient Category"].value_counts().head(10).to_string())

In [ ]:
# ── METRIC: Units at risk (P90 > 300s, ≥20 calls) ──────────────────────────
# Formula: units where response P90 > 300s AND call count >= 20
# Denominator: units with >=20 calls (sparse units excluded to avoid skew)
unit_stats = (
    df[rt_valid].groupby("Unit")["total_response_s"]
    .agg(calls="count", p90=lambda x: x.quantile(0.90))
)
qualified = unit_stats[unit_stats["calls"] >= 20]
at_risk    = qualified[qualified["p90"] > 300]
print(f"Units with ≥20 calls (qualified denominator): {len(qualified)}")
print(f"Units at risk (P90 > 300s): {len(at_risk)}")
print()
print("All qualified units:")
print(qualified.sort_values('p90', ascending=False).to_string())

## 7. Caveats

Document every limitation that affects interpretation. Update this list when new issues are found.

### 7.1 Minute-resolution timestamps
All timestamps in this file are minute-precision (`HH:MM`, seconds always `:00`).  
**Effect:** Phase intervals can only be measured to ±60s precision.  
A "1-minute call processing time" could be anywhere from 1 second to 119 seconds.  
**Mitigation:** Use medians not means (less sensitive to rounding). Do not report sub-minute intervals.

### 7.2 Cancelled calls with no arrival time
~N% of calls have no `Date Arrived`. These are excluded from ALL response-time metrics.  
**Effect:** Response time denominators are smaller than total call volume.  
**Dashboard must state:** "Response time metrics computed on calls with recorded arrival. N cancelled/refused calls excluded."

### 7.3 Patient Category vs Call Type
`Call Type` is largely null. `Patient Category` is used as incident type.  
These are not the same field. `Patient Category` reflects the **outcome classification**, not the dispatch reason.  
**Effect:** Incident type breakdown reflects what paramedics found, not what was reported by caller.

### 7.4 Unit naming convention
Units are named `BLS####`. This file contains only BLS units.  
If ALS units or mutual aid units exist in other files, they must be explicitly labeled in any multi-file analysis.

### 7.5 Date Arrive Rec vs Date Arrived
Two fields with similar names exist: `Date Arrive Rec` and `Date Arrived`.  
Based on row inspection, `Date Arrive Rec` appears to be the **requested** arrival time (often null).  
`Date Arrived` is the actual on-scene timestamp and is used for all response time calculations.  
**Confirm with agency what `Date Arrive Rec` represents before using it.**

### 7.6 What this data cannot tell you
- It cannot tell you why a unit was late (traffic, simultaneous call, post positioning)
- It cannot tell you whether a cancelled call was a false alarm or a true emergency
- It cannot distinguish ALS-capable BLS units from non-ALS units
- It cannot measure call-taker processing time (data starts at dispatch, not call receipt... wait, check `Date Received`)